# Practice work # 3

> Student name    - Rodion

> Student surname - Ostapenko

> Group           - CS-32

### Task

Напишіть своб власну реалізацію лінійної регресії для набору ознак із регуляризацією L1 та L2. Покажіть особливості роботи на синтетичному наборі даних.
До Гугл-класу прикріпити Jupiter Notebook, де проведені вказані обчислення та текстові комірки для оформлення та пояснення дій і результатів.

In [106]:
import numpy as np
SEED = 20

n_features = 3
n_samples = 100

In [107]:
def generate_linear_data(n_features, n_samples, noise=5):
    np.random.seed(SEED)
    true_bias = np.random.uniform(-10, 10)
    true_weights = np.random.uniform(-10, 10, size=n_features)
    
    X = np.random.rand(n_samples, n_features) * 10 - 5
    y = true_bias + (true_weights @ X.T)
    
    y_noised = y + np.random.normal(scale=noise, size=n_samples)
    
    return X, y_noised, [true_bias, true_weights]

In [108]:
data = generate_linear_data(n_features, n_samples)
true_weights = data[2]

In [109]:
def print_compare(calculated_weights, real_weights, model_name=None):
    bias = calculated_weights[0]
    weights = calculated_weights[1:]
    true_bias, true_weights = real_weights
    if model_name is not None:
        print(f"---- {model_name} ----")
    print(f"Обчислене відхилення: {bias:.8f}, Реальне відхилення: {true_bias:.8f} | Різниця: {abs(true_bias-bias):.8f}")
    for w, true_w in zip(weights, true_weights):
        print(f"Обчислені ваги: {w:.8f}, Реальні ваги: {true_w:.8f} | Різниця: {abs(w-true_w):.8f}")

### L-2 Regularization
$$J(\theta) = \text{MSE}(\theta) + \frac{\alpha}{m} \sum_{i=1}^{n} \theta_i^2$$

### L-1 Regularization
$$J(\mathbf{\theta}) = \text{MSE}(\mathbf{\theta}) + \alpha \sum_{i=1}^{n} \vert{}\theta_i\vert{}$$

#### Для розрахунку градієнту будуть використовуватися похідні цих формул, тобто $2\theta$ для L-2 та $sign(\theta)$ для L-1

In [110]:
def mse(y_true, X, theta):
    m = len(y_true)
    MSE = (1/m) * sum((X @ theta - y_true)** 2) 
    return MSE

def mse_gradient(y_true, X, theta, regularization, regularization_hyperparameter):
    m = len(y_true)
    gradient_loss = (2/m) * (X.T @ (X @ theta - y_true))
    theta_reg = theta.copy()
    theta_reg[0] = 0
    
    if regularization == "l2":
        addition = (regularization_hyperparameter/m) * 2 * theta_reg
    elif regularization == "l1":
        addition = regularization_hyperparameter * np.sign(theta_reg)
    else:
        addition= 0
    gradient = gradient_loss + addition
    
    return gradient

def batch_gradient_descent(X, y, epoches=101, learning_rate=0.1, regularization=None, rh=0):
    X_with_bias = np.column_stack((np.ones(n_samples), X))
    current_weights = np.random.uniform(-1, 1, size=n_features+1)
    
    for i in range(epoches):
        MSE = mse(y, X_with_bias, current_weights)
        if i % 20 == 0:
            print(f"Epoch {i}: MSE {MSE:.8f} | Weights: {current_weights}")
            
        error = mse_gradient(y, X_with_bias, current_weights, regularization, rh)
        
        current_weights -= learning_rate * error
    return current_weights

In [111]:
results = batch_gradient_descent(data[0], data[1])
print_compare(results, true_weights)

Epoch 0: MSE 1367.75387687 | Weights: [-0.19422404  0.62092484 -0.8016      0.93008158]
Epoch 20: MSE 19.91698528 | Weights: [2.55222609 8.17941567 7.69852871 6.05706898]
Epoch 40: MSE 19.91530366 | Weights: [2.58611984 8.18588614 7.70210065 6.05351605]
Epoch 60: MSE 19.91530324 | Weights: [2.58678566 8.18590024 7.70205556 6.05345054]
Epoch 80: MSE 19.91530324 | Weights: [2.58679886 8.18590042 7.70205458 6.05344925]
Epoch 100: MSE 19.91530324 | Weights: [2.58679912 8.18590042 7.70205456 6.05344923]
Обчислене відхилення: 2.58679912, Реальне відхилення: 1.76261602 | Різниця: 0.82418310
Обчислені ваги: 8.18590043, Реальні ваги: 7.95427456 | Різниця: 0.23162587
Обчислені ваги: 7.70205456, Реальні ваги: 7.83061459 | Різниця: 0.12856003
Обчислені ваги: 6.05344923, Реальні ваги: 6.31674955 | Різниця: 0.26330032


In [112]:
results = batch_gradient_descent(data[0], data[1], learning_rate=0.01, regularization="l2", rh=1000)
print_compare(results, true_weights)

Epoch 0: MSE 1297.11449583 | Weights: [0.73938327 0.06578812 0.28051557 0.6102196 ]
Epoch 20: MSE 421.42051479 | Weights: [2.85842728 3.67988675 3.58899431 2.62993867]
Epoch 40: MSE 417.79804029 | Weights: [4.18292056 3.68761893 3.54753056 2.57603454]
Epoch 60: MSE 417.24390267 | Weights: [5.08402989 3.69257755 3.51914815 2.53912766]
Epoch 80: MSE 417.72950044 | Weights: [5.6971021  3.69595111 3.4998381  2.51401792]
Epoch 100: MSE 418.45915894 | Weights: [6.11420747 3.69824632 3.48670045 2.49693444]
Обчислене відхилення: 6.13114020, Реальне відхилення: 1.76261602 | Різниця: 4.36852418
Обчислені ваги: 3.69833950, Реальні ваги: 7.95427456 | Різниця: 4.25593506
Обчислені ваги: 3.48616712, Реальні ваги: 7.83061459 | Різниця: 4.34444747
Обчислені ваги: 2.49624093, Реальні ваги: 6.31674955 | Різниця: 3.82050862


In [113]:
results = batch_gradient_descent(data[0], data[1], learning_rate=0.001, regularization="l1", rh=10)
print_compare(results, true_weights)

Epoch 0: MSE 1514.35056974 | Weights: [-0.02137791 -0.38583355 -0.64116452  0.46794448]
Epoch 20: MSE 816.56354178 | Weights: [0.3401076  1.90028225 1.77004001 1.84128665]
Epoch 40: MSE 461.53081807 | Weights: [0.61957076 3.49355755 3.39372364 2.84033476]
Epoch 60: MSE 271.99585695 | Weights: [0.83985618 4.63577845 4.53576672 3.56687589]
Epoch 80: MSE 169.14788991 | Weights: [1.01710165 5.45517089 5.33807866 4.09494515]
Epoch 100: MSE 112.20712968 | Weights: [1.16287419 6.0433878  5.90090335 4.47844205]
Обчислене відхилення: 1.16949674, Реальне відхилення: 1.76261602 | Різниця: 0.59311928
Обчислені ваги: 6.06799572, Реальні ваги: 7.95427456 | Різниця: 1.88627883
Обчислені ваги: 5.92412829, Реальні ваги: 7.83061459 | Різниця: 1.90648630
Обчислені ваги: 4.49457690, Реальні ваги: 6.31674955 | Різниця: 1.82217264


In [114]:
results = batch_gradient_descent(data[0], data[1], learning_rate=0.001, regularization="l1", rh=100)
print_compare(results, true_weights)

Epoch 0: MSE 1334.61239545 | Weights: [ 0.38912968  0.05393028  0.84353323 -0.45777984]
Epoch 20: MSE 1162.07536130 | Weights: [0.75472287 0.60163369 1.23147899 0.04396803]
Epoch 40: MSE 1071.20955036 | Weights: [1.09956309 0.99472423 1.49755369 0.12613612]
Epoch 60: MSE 1025.15125731 | Weights: [1.42782482 1.27663129 1.67849754 0.00672257]
Epoch 80: MSE 979.45025981 | Weights: [1.73964535 1.48004934 1.79902639 0.0664572 ]
Epoch 100: MSE 945.43458544 | Weights: [2.03843825 1.62652859 1.87780755 0.12729663]
Обчислене відхилення: 2.05297938, Реальне відхилення: 1.76261602 | Різниця: 0.29036336
Обчислені ваги: 1.63270760, Реальні ваги: 7.95427456 | Різниця: 6.32156695
Обчислені ваги: 1.88086925, Реальні ваги: 7.83061459 | Різниця: 5.94974534
Обчислені ваги: 0.11948802, Реальні ваги: 6.31674955 | Різниця: 6.19726152


Базова модель досягає найкращих результатів та найбільш точно наближає ваги до реальних значень. У цьому випадку модель добре описує дані без додаткових обмежень.

Надмірна L2-регуляризація високий коефіцієнт штрафу змушує модель занадто сильно зменшувати ваги. Це призводить до жорсткого недонавчання

L-1 При помірному штрафі все ще намагається зійтися, хоча ваги занижені. При високому штрафі проявляється ключова властивість L-2 регуляризації - розрядженість. Окрім загального недонавчання, остання вага агресивно стягується майже до нуля.

Особливості роботи з синтетичним набором даних:
* Точне знання реальних параметрів для еталонної перевірки результатів моделі.
* Відсутність непередбачуваних пропусків, помилок збору.
* Відірваність від реальності. Синтетика не відтворює повної хаотичності реальних даних, що створює ризик переоцінки ефективності алгоритму.